In [1]:
!pip install -q rdkit lightgbm huggingface_hub
import pandas as pd, numpy as np
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id="openadmet/cyp-challenge-train-test",
                         repo_type="dataset")
import os
for root, _, files in os.walk(path):
    for f in files:
        full = os.path.join(root, f)
        print(os.path.relpath(full, path), os.path.getsize(full)//1024, "KB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 34.1 MB/s eta 0:00:00


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

cyp-challenge-TEST-BLINDED_structures.csv 1 KB
cyp-challenge-TEST-BLINDED.csv 43 KB
cyp-challenge-TRAIN_TDI.csv 1189 KB
README.md 4 KB
.gitattributes 2 KB
cyp-challenge-TRAIN_inhibition.csv 638 KB
cyp-challenge-single-concentration-TRAIN.csv 3534 KB
cyp-challenge-TRAIN_Emax.csv 1104 KB


In [2]:
print(open(os.path.join(path, "README.md")).read())
print("="*80)

files = ["cyp-challenge-TRAIN_inhibition.csv",
         "cyp-challenge-single-concentration-TRAIN.csv",
         "cyp-challenge-TRAIN_Emax.csv",
         "cyp-challenge-TRAIN_TDI.csv",
         "cyp-challenge-TEST-BLINDED.csv"]
dfs = {}
for f in files:
    d = pd.read_csv(os.path.join(path, f))
    dfs[f] = d
    print("\n###", f, d.shape)
    print(list(d.columns))
    print(d.head(2).T.to_string())

---
license: apache-2.0
language:
- en
tags:
- chemistry
- drug-discovery
- ADMET
- molecular-properties
- blind-challenge
- computational-chemistry
- cytochrome-p450

pretty_name: OpenADMET CYP Inhibition Blind Challenge
size_categories:
- 10K<n<100K
task_categories:
- tabular-regression
annotations_creators:
- expert-generated
source_datasets:
- original
configs:
- config_name: default
  data_files:
  - split: train
    path: cyp-challenge-TRAIN_inhibition.csv
  - split: test
    path: cyp-challenge-TEST-BLINDED.csv
- config_name: tdi
  data_files:
  - split: train
    path: cyp-challenge-TRAIN_TDI.csv
- config_name: single_concentration
  data_files:
  - split: train
    path: cyp-challenge-single-concentration-TRAIN.csv
- config_name: emax
  data_files:
  - split: train
    path: cyp-challenge-TRAIN_Emax.csv
- config_name: structure
  data_files:
  - split: test
    path: cyp-challenge-TEST-BLINDED_structures.csv
---

# CYP Challenge Train/Test Dataset

A high-quality experimental 

In [3]:
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from scipy.stats import spearmanr
RDLogger.DisableLog('rdApp.*')

tr = dfs["cyp-challenge-TRAIN_inhibition.csv"]
sc = dfs["cyp-challenge-single-concentration-TRAIN.csv"]
te = dfs["cyp-challenge-TEST-BLINDED.csv"]
iso = ["CYP1A2", "CYP2C9", "CYP2D6", "CYP3A4"]
pc = {i: f"{i}_pIC50_direct_inhibition" for i in iso}

print("1) pIC50 labels per isoform")
for i in iso:
    s = tr[pc[i]].dropna()
    print(f"  {i}: n={len(s)}  min={s.min():.2f}  median={s.median():.2f}  max={s.max():.2f}  frac<3={np.mean(s<3):.2f}")

print("\n2) isoforms measured per training compound:",
      tr[list(pc.values())].notna().sum(axis=1).value_counts().sort_index().to_dict())

print("\n3) single-conc: compounds =", sc.Molecule_Name.nunique(),
      "| rows per enzyme =", sc.enzyme.value_counts().to_dict(),
      "| concentrations =", sc.concentration_M.unique())

a, b, t = set(tr.Molecule_Name), set(sc.Molecule_Name), set(te.Molecule_Name)
print("\n4) overlap: DRC compounds also in single-conc =", len(a & b), "of", len(a),
      "| single-conc compounds with no DRC =", len(b - a),
      "| test compounds already in train/single =", len(t & (a | b)))

print("\n5) Spearman(single-conc log2fc, pIC50) per isoform")
wide = sc.pivot_table(index="Molecule_Name", columns="enzyme", values="log2fc_estimate")
m = tr.set_index("Molecule_Name").join(wide, rsuffix="_fc")
for i in iso:
    d = m[[pc[i], i]].dropna()
    print(f"  {i}: n={len(d)}  rho={spearmanr(d[pc[i]], d[i])[0]:.2f}")

print("\n6) nearest-neighbour Tanimoto (Morgan r2)")
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
def fps(smiles):
    out = []
    for s in smiles:
        mol = Chem.MolFromSmiles(s)
        if mol is not None:
            out.append(gen.GetFingerprint(mol))
    return out
pool = pd.concat([tr[["Molecule_Name","SMILES"]], sc[["Molecule_Name","SMILES"]]]).drop_duplicates("Molecule_Name")
pool_fp = fps(pool.SMILES)
test_fp = fps(te.SMILES)
nn_test = [max(DataStructs.BulkTanimotoSimilarity(f, pool_fp)) for f in test_fp]
rng = np.random.default_rng(0)
idx = rng.choice(len(pool_fp), 500, replace=False)
nn_train = [sorted(DataStructs.BulkTanimotoSimilarity(pool_fp[i], pool_fp))[-2] for i in idx]
q = [10, 50, 90]
print("  test -> train/pool  percentiles 10/50/90:", np.percentile(nn_test, q).round(2))
print("  train -> train/pool percentiles 10/50/90:", np.percentile(nn_train, q).round(2))

1) pIC50 labels per isoform
  CYP1A2: n=1412  min=1.91  median=5.13  max=7.95  frac<3=0.06
  CYP2C9: n=1285  min=2.10  median=4.62  max=7.47  frac<3=0.04
  CYP2D6: n=1493  min=1.95  median=4.73  max=7.53  frac<3=0.06
  CYP3A4: n=2335  min=1.91  median=4.27  max=7.19  frac<3=0.19

2) isoforms measured per training compound: {1: 3596, 2: 1039, 3: 229, 4: 41}

3) single-conc: compounds = 4376 | rows per enzyme = {'CYP1A2': 4376, 'CYP2C9': 4376, 'CYP2D6': 4376, 'CYP3A4': 4376} | concentrations = [4.95049505e-05]

4) overlap: DRC compounds also in single-conc = 4375 of 4905 | single-conc compounds with no DRC = 1 | test compounds already in train/single = 0

5) Spearman(single-conc log2fc, pIC50) per isoform
  CYP1A2: n=1412  rho=-0.86
  CYP2C9: n=1285  rho=-0.90
  CYP2D6: n=1493  rho=-0.83
  CYP3A4: n=1805  rho=-0.94

6) nearest-neighbour Tanimoto (Morgan r2)
  test -> train/pool  percentiles 10/50/90: [0.52 0.59 0.7 ]
  train -> train/pool percentiles 10/50/90: [0.34 0.45 0.6 ]


In [4]:
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import KFold

iso_of = {v: k for k, v in pc.items()}
lab = (tr.melt(id_vars="Molecule_Name", value_vars=list(pc.values()),
               var_name="col", value_name="pIC50").dropna())
lab["enzyme"] = lab.col.map(iso_of)
allp = sc[["Molecule_Name", "enzyme", "log2fc_estimate"]]
lab = lab.merge(allp, on=["Molecule_Name", "enzyme"], how="left")
unl = allp.merge(lab[["Molecule_Name", "enzyme"]], on=["Molecule_Name", "enzyme"],
                 how="left", indicator=True)
unl = unl[unl["_merge"] == "left_only"]
print("labeled pairs:", len(lab), "| with screen data:", lab.log2fc_estimate.notna().sum(),
      "| unlabeled pairs:", len(unl))

print("\nA) log2fc percentiles 5/50/95, labeled vs unlabeled pairs")
for i in iso:
    l = lab[lab.enzyme == i].log2fc_estimate.dropna()
    u = unl[unl.enzyme == i].log2fc_estimate
    print(f"  {i}: labeled {np.percentile(l,[5,50,95]).round(2)} | unlabeled {np.percentile(u,[5,50,95]).round(2)}")

print("\nB) out-of-fold isotonic map log2fc -> pIC50")
for i in iso:
    d = lab[lab.enzyme == i].dropna(subset=["log2fc_estimate"]).reset_index(drop=True)
    oof = np.zeros(len(d))
    for a_, b_ in KFold(5, shuffle=True, random_state=0).split(d):
        ir = IsotonicRegression(increasing=False, out_of_bounds="clip").fit(
            d.log2fc_estimate[a_], d.pIC50[a_])
        oof[b_] = ir.predict(d.log2fc_estimate[b_])
    print(f"  {i}: n={len(d)}  MAE={np.mean(np.abs(oof - d.pIC50)):.2f}  rho={spearmanr(oof, d.pIC50)[0]:.2f}")

print("\nC) training compounds with no screen data")
only = tr[~tr.Molecule_Name.isin(sc.Molecule_Name)]
print("  n =", len(only), "| labels per isoform:", only[list(pc.values())].notna().sum().to_dict())
print("  CYP3A4 pIC50 summary:", only["CYP3A4_pIC50_direct_inhibition"].describe().round(2).to_dict())
print("  example SMILES:", only.SMILES.head(5).tolist())

labeled pairs: 6525 | with screen data: 5995 | unlabeled pairs: 11509

A) log2fc percentiles 5/50/95, labeled vs unlabeled pairs
  CYP1A2: labeled [-1.97 -1.55 -0.19] | unlabeled [-1.15 -0.2   0.08]
  CYP2C9: labeled [-1.37 -0.81 -0.19] | unlabeled [-1.36 -0.6   0.01]
  CYP2D6: labeled [-2.95 -1.74 -0.73] | unlabeled [-0.32  0.38  0.61]
  CYP3A4: labeled [-4.03 -1.44 -0.01] | unlabeled [-4.16 -1.93 -0.2 ]

B) out-of-fold isotonic map log2fc -> pIC50
  CYP1A2: n=1412  MAE=0.31  rho=0.85
  CYP2C9: n=1285  MAE=0.29  rho=0.89
  CYP2D6: n=1493  MAE=0.45  rho=0.82
  CYP3A4: n=1805  MAE=0.32  rho=0.93

C) training compounds with no screen data
  n = 530 | labels per isoform: {'CYP1A2_pIC50_direct_inhibition': 0, 'CYP2C9_pIC50_direct_inhibition': 0, 'CYP2D6_pIC50_direct_inhibition': 0, 'CYP3A4_pIC50_direct_inhibition': 530}
  CYP3A4 pIC50 summary: {'count': 530.0, 'mean': 4.31, 'std': 1.0, 'min': 1.94, '25%': 3.81, '50%': 4.45, '75%': 4.99, 'max': 6.8}
  example SMILES: ['N#CC1=CC(C(F)(F)F)=CC

In [5]:
from rdkit.Chem import Descriptors
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.isotonic import IsotonicRegression

U = tr.drop_duplicates("Molecule_Name").reset_index(drop=True)
mols = [Chem.MolFromSmiles(s) for s in U.SMILES]
keep = [m is not None for m in mols]
print("unparsed SMILES dropped:", len(keep) - sum(keep))
U = U[keep].reset_index(drop=True); mols = [m for m in mols if m is not None]

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fp_counts = np.array([gen.GetCountFingerprintAsNumPy(m) for m in mols], dtype=np.float32)
desc = np.array([[f(m) for _, f in Descriptors.descList] for m in mols], dtype=np.float64)
desc = np.clip(np.nan_to_num(desc, nan=0, posinf=1e6, neginf=-1e6), -1e6, 1e6).astype(np.float32)
X = np.hstack([fp_counts, desc])
print("feature matrix:", X.shape)

bv = [gen.GetFingerprint(m) for m in mols]
nn = np.zeros(len(U))
for k, f in enumerate(bv):
    s = np.array(DataStructs.BulkTanimotoSimilarity(f, bv)); s[k] = 0
    nn[k] = s.max()

wide = sc.pivot_table(index="Molecule_Name", columns="enzyme", values="log2fc_estimate")
y  = {i: U[pc[i]].values for i in iso}
fc = {i: U.Molecule_Name.map(wide[i]).values for i in iso}

params = dict(n_estimators=300, learning_rate=0.05, num_leaves=31,
              colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
              min_child_samples=10, n_jobs=-1, verbose=-1)
kf = KFold(5, shuffle=True, random_state=42)
rows = []
for i in iso:
    yi, fci = y[i], fc[i]
    P = {k: np.full(len(U), np.nan) for k in ["A_labeled_only", "B_plus_soft"]}
    for tr_idx, te_idx in kf.split(U):
        lab = tr_idx[~np.isnan(yi[tr_idx])]
        P["A_labeled_only"][te_idx] = lgb.LGBMRegressor(**params).fit(X[lab], yi[lab]).predict(X[te_idx])
        ok = lab[~np.isnan(fci[lab])]
        ir = IsotonicRegression(increasing=False, out_of_bounds="clip").fit(fci[ok], yi[ok])
        unl = tr_idx[np.isnan(yi[tr_idx]) & ~np.isnan(fci[tr_idx])]
        Xs = np.vstack([X[lab], X[unl]])
        ys = np.concatenate([yi[lab], ir.predict(fci[unl])])
        w = np.concatenate([np.ones(len(lab)), np.full(len(unl), 0.5)])
        P["B_plus_soft"][te_idx] = lgb.LGBMRegressor(**params).fit(Xs, ys, sample_weight=w).predict(X[te_idx])
    for name, p in P.items():
        for subset, msk in [("all", np.ones(len(U), bool)), ("nn>=0.5", nn >= 0.5)]:
            a = msk & ~np.isnan(fci)
            b = msk & ~np.isnan(yi)
            rows.append(dict(isoform=i, model=name, subset=subset,
                             screen_rho=spearmanr(p[a], -fci[a])[0],
                             labeled_MAE=np.mean(np.abs(p[b] - yi[b])),
                             labeled_rho=spearmanr(p[b], yi[b])[0]))
    print(i, "done")

R = pd.DataFrame(rows).round(3)
print(R.to_string(index=False))
print("\nmean over isoforms:")
print(R.groupby(["subset", "model"])[["screen_rho", "labeled_MAE", "labeled_rho"]].mean().round(3))

unparsed SMILES dropped: 0
feature matrix: (4905, 2265)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/v

CYP1A2 done


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/v

CYP2C9 done


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/v

CYP2D6 done


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/v

CYP3A4 done
isoform          model  subset  screen_rho  labeled_MAE  labeled_rho
 CYP1A2 A_labeled_only     all       0.525        0.665        0.510
 CYP1A2 A_labeled_only nn>=0.5       0.476        0.755        0.532
 CYP1A2    B_plus_soft     all       0.628        0.849        0.531
 CYP1A2    B_plus_soft nn>=0.5       0.625        0.814        0.606
 CYP2C9 A_labeled_only     all       0.665        0.482        0.585
 CYP2C9 A_labeled_only nn>=0.5       0.613        0.500        0.542
 CYP2C9    B_plus_soft     all       0.727        0.481        0.660
 CYP2C9    B_plus_soft nn>=0.5       0.708        0.497        0.619
 CYP2D6 A_labeled_only     all       0.302        0.621        0.341
 CYP2D6 A_labeled_only nn>=0.5       0.339        0.615        0.384
 CYP2D6    B_plus_soft     all       0.454        0.723        0.414
 CYP2D6    B_plus_soft nn>=0.5       0.470        0.703        0.452
 CYP3A4 A_labeled_only     all       0.701        0.539        0.768
 CYP3A4 A_labeled_only

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [6]:
import warnings; warnings.filterwarnings("ignore")

def st_rae(y, p, lo, hi):
    y, p, lo, hi = map(np.asarray, (y, p, lo, hi))
    err = np.clip(p - hi, 0, None) + np.clip(lo - p, 0, None)
    m = y.mean()
    base = np.clip(m - hi, 0, None) + np.clip(lo - m, 0, None)
    return err.sum() / base.sum()

lo_col = {i: U[f"{pc[i]}_conf_low"].values for i in iso}
hi_col = {i: U[f"{pc[i]}_conf_high"].values for i in iso}

# pseudo-truth for unlabeled pairs (evaluation only)
pseudo = {}
for i in iso:
    yi, fci = y[i], fc[i]
    labd = ~np.isnan(yi) & ~np.isnan(lo_col[i]) & ~np.isnan(hi_col[i])
    ok = labd & ~np.isnan(fci)
    irf = IsotonicRegression(increasing=False, out_of_bounds="clip").fit(fci[ok], yi[ok])
    edges = np.quantile(yi[labd], np.linspace(0, 1, 11))
    b = np.clip(np.digitize(yi[labd], edges[1:-1]), 0, 9)
    dlo = np.array([np.median((yi[labd] - lo_col[i][labd])[b == k]) for k in range(10)])
    dhi = np.array([np.median((hi_col[i][labd] - yi[labd])[b == k]) for k in range(10)])
    unl = np.isnan(yi) & ~np.isnan(fci)
    pt = irf.predict(fci[unl]); bu = np.clip(np.digitize(pt, edges[1:-1]), 0, 9)
    t = np.where(labd, yi, np.nan); l = np.where(labd, lo_col[i], np.nan); h = np.where(labd, hi_col[i], np.nan)
    t[unl] = pt; l[unl] = pt - dlo[bu]; h[unl] = pt + dhi[bu]
    pseudo[i] = (t, l, h)

cfgs = {"A_labeled_only": None, "B_soft_w0.5": 0.5, "B_soft_w1.0": 1.0}
OOF = {m: {i: np.full(len(U), np.nan) for i in iso} for m in cfgs}
for i in iso:
    yi, fci = y[i], fc[i]
    for tr_idx, te_idx in kf.split(U):
        lab = tr_idx[~np.isnan(yi[tr_idx])]
        okf = lab[~np.isnan(fci[lab])]
        ir = IsotonicRegression(increasing=False, out_of_bounds="clip").fit(fci[okf], yi[okf])
        unl_tr = tr_idx[np.isnan(yi[tr_idx]) & ~np.isnan(fci[tr_idx])]
        Xs = np.vstack([X[lab], X[unl_tr]])
        ys = np.concatenate([yi[lab], ir.predict(fci[unl_tr])])
        for name, wt in cfgs.items():
            if wt is None:
                m_ = lgb.LGBMRegressor(**params).fit(X[lab], yi[lab])
            else:
                w = np.concatenate([np.ones(len(lab)), np.full(len(unl_tr), wt)])
                m_ = lgb.LGBMRegressor(**params).fit(Xs, ys, sample_weight=w)
            OOF[name][i][te_idx] = m_.predict(X[te_idx])
    print(i, "done")

rows = []
for name in cfgs:
    for i in iso:
        p = OOF[name][i]; yi = y[i]
        labd = ~np.isnan(yi) & ~np.isnan(lo_col[i]) & ~np.isnan(hi_col[i])
        t, l, h = pseudo[i]; m = ~np.isnan(t)
        rows.append(dict(model=name, isoform=i,
                         ST_RAE_labeled=st_rae(yi[labd], p[labd], lo_col[i][labd], hi_col[i][labd]),
                         ST_RAE_pseudo=st_rae(t[m], p[m], l[m], h[m]),
                         n_pseudo=int(m.sum())))
R2 = pd.DataFrame(rows).round(3)
print(R2.to_string(index=False))
print("\nmacro mean (lower is better):")
print(R2.groupby("model")[["ST_RAE_labeled", "ST_RAE_pseudo"]].mean().round(3))

CYP1A2 done
CYP2C9 done
CYP2D6 done
CYP3A4 done
         model isoform  ST_RAE_labeled  ST_RAE_pseudo  n_pseudo
A_labeled_only  CYP1A2           0.848          0.874      4375
A_labeled_only  CYP2C9           0.695          0.573      4375
A_labeled_only  CYP2D6           0.985          0.638      4375
A_labeled_only  CYP3A4           0.511          0.581      4905
   B_soft_w0.5  CYP1A2           1.281          0.606      4375
   B_soft_w0.5  CYP2C9           0.782          0.493      4375
   B_soft_w0.5  CYP2D6           1.235          0.781      4375
   B_soft_w0.5  CYP3A4           0.479          0.518      4905
   B_soft_w1.0  CYP1A2           1.331          0.617      4375
   B_soft_w1.0  CYP2C9           0.786          0.493      4375
   B_soft_w1.0  CYP2D6           1.307          0.827      4375
   B_soft_w1.0  CYP3A4           0.474          0.510      4905

macro mean (lower is better):
                ST_RAE_labeled  ST_RAE_pseudo
model                                      

In [7]:
rows = []
for i in iso:
    yi, fci = y[i], fc[i]
    lab = ~np.isnan(yi)
    inact = np.isnan(yi) & (fci > -0.3)
    weak = lab & (yi < 4)
    pot = lab & (yi >= 5)
    for name in OOF:
        p = OOF[name][i]
        rows.append({
            "isoform": i, "model": name,
            "n_clear_inactive": int(inact.sum()),
            "inact_mean_pred": p[inact].mean(),
            "inact_frac_pred_above_3.5": (p[inact] > 3.5).mean(),
            "labeled_bias": (p[lab] - yi[lab]).mean(),
            "MAE_weak(<4)": np.abs(p[weak] - yi[weak]).mean(),
            "MAE_potent(>=5)": np.abs(p[pot] - yi[pot]).mean(),
        })
D = pd.DataFrame(rows).round(3)
print(D.to_string(index=False))

isoform          model  n_clear_inactive  inact_mean_pred  inact_frac_pred_above_3.5  labeled_bias  MAE_weak(<4)  MAE_potent(>=5)
 CYP1A2 A_labeled_only              1777            4.309                      0.918        -0.013         1.351            0.574
 CYP1A2    B_soft_w0.5              1777            3.331                      0.361        -0.572         0.744            0.966
 CYP1A2    B_soft_w1.0              1777            3.300                      0.335        -0.619         0.712            1.005
 CYP2C9 A_labeled_only              1019            4.012                      0.869        -0.000         0.720            0.583
 CYP2C9    B_soft_w0.5              1019            3.570                      0.546        -0.203         0.519            0.649
 CYP2C9    B_soft_w1.0              1019            3.551                      0.542        -0.215         0.522            0.650
 CYP2D6 A_labeled_only              2720            4.523                      0.994      

In [8]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

OOF["C_expected"] = {i: np.full(len(U), np.nan) for i in iso}
OOF["C_hard"]     = {i: np.full(len(U), np.nan) for i in iso}
auc = {}
for i in iso:
    yi, fci = y[i], fc[i]
    pinh = np.full(len(U), np.nan)
    for tr_idx, te_idx in kf.split(U):
        scr = tr_idx[~np.isnan(fci[tr_idx])]
        clf = LGBMClassifier(**params).fit(X[scr], (fci[scr] <= -0.3).astype(int))
        p_inh = clf.predict_proba(X[te_idx])[:, 1]
        lab = tr_idx[~np.isnan(yi[tr_idx])]
        weak = lab[(~np.isnan(fci[lab])) & (fci[lab] > -0.3)]
        floor = np.median(yi[weak]) if len(weak) >= 10 else np.percentile(yi[lab], 10)
        reg = OOF["A_labeled_only"][i][te_idx]
        pinh[te_idx] = p_inh
        OOF["C_expected"][i][te_idx] = floor + p_inh * (reg - floor)
        OOF["C_hard"][i][te_idx] = np.where(p_inh > 0.5, reg, floor)
    m = ~np.isnan(fci)
    auc[i] = roc_auc_score(fci[m] <= -0.3, pinh[m])
    print(i, "done  | classifier AUC =", round(auc[i], 3))

rows, diag = [], []
for name in ["A_labeled_only", "B_soft_w0.5", "C_expected", "C_hard"]:
    for i in iso:
        p = OOF[name][i]; yi, fci = y[i], fc[i]
        labd = ~np.isnan(yi) & ~np.isnan(lo_col[i]) & ~np.isnan(hi_col[i])
        t, l, h = pseudo[i]; mm = ~np.isnan(t)
        rows.append(dict(model=name, isoform=i,
                         ST_RAE_labeled=st_rae(yi[labd], p[labd], lo_col[i][labd], hi_col[i][labd]),
                         ST_RAE_pseudo=st_rae(t[mm], p[mm], l[mm], h[mm])))
        lab = ~np.isnan(yi); inact = np.isnan(yi) & (fci > -0.3); pot = lab & (yi >= 5)
        diag.append(dict(model=name, isoform=i,
                         inact_mean_pred=p[inact].mean(),
                         frac_inact_above_3p5=(p[inact] > 3.5).mean(),
                         labeled_bias=(p[lab] - yi[lab]).mean(),
                         MAE_potent=np.abs(p[pot] - yi[pot]).mean()))
R3 = pd.DataFrame(rows).round(3); D3 = pd.DataFrame(diag).round(3)
print("\n", R3.to_string(index=False))
print("\nmacro mean (lower is better):")
print(R3.groupby("model")[["ST_RAE_labeled", "ST_RAE_pseudo"]].mean().round(3))
print("\n", D3.to_string(index=False))

CYP1A2 done  | classifier AUC = 0.814
CYP2C9 done  | classifier AUC = 0.87
CYP2D6 done  | classifier AUC = 0.806
CYP3A4 done  | classifier AUC = 0.879

          model isoform  ST_RAE_labeled  ST_RAE_pseudo
A_labeled_only  CYP1A2           0.848          0.874
A_labeled_only  CYP2C9           0.695          0.573
A_labeled_only  CYP2D6           0.985          0.638
A_labeled_only  CYP3A4           0.511          0.581
   B_soft_w0.5  CYP1A2           1.281          0.606
   B_soft_w0.5  CYP2C9           0.782          0.493
   B_soft_w0.5  CYP2D6           1.235          0.781
   B_soft_w0.5  CYP3A4           0.479          0.518
    C_expected  CYP1A2           1.100          0.605
    C_expected  CYP2C9           0.727          0.572
    C_expected  CYP2D6           1.040          0.660
    C_expected  CYP3A4           0.544          0.609
        C_hard  CYP1A2           1.053          0.735
        C_hard  CYP2C9           0.722          0.576
        C_hard  CYP2D6           1.12

In [9]:
PINH  = {i: np.full(len(U), np.nan) for i in iso}
FLOOR = {i: np.full(len(U), np.nan) for i in iso}
for i in iso:
    yi, fci = y[i], fc[i]
    for tr_idx, te_idx in kf.split(U):
        scr = tr_idx[~np.isnan(fci[tr_idx])]
        clf = LGBMClassifier(**params).fit(X[scr], (fci[scr] <= -0.3).astype(int))
        PINH[i][te_idx] = clf.predict_proba(X[te_idx])[:, 1]
        lab_y = yi[tr_idx][~np.isnan(yi[tr_idx])]
        low = lab_y[lab_y < 3.0]
        FLOOR[i][te_idx] = np.median(low) if len(low) >= 10 else 2.2
    print(i, "done | mean floor =", round(np.nanmean(FLOOR[i]), 2))

OOF["D_exp_lowfloor"] = {i: FLOOR[i] + PINH[i] * (OOF["A_labeled_only"][i] - FLOOR[i]) for i in iso}
OOF["E_avg_A_B"]      = {i: 0.5 * (OOF["A_labeled_only"][i] + OOF["B_soft_w0.5"][i]) for i in iso}
OOF["F_avg_A_D"]      = {i: 0.5 * (OOF["A_labeled_only"][i] + OOF["D_exp_lowfloor"][i]) for i in iso}

rows = []
for name in ["A_labeled_only", "B_soft_w0.5", "D_exp_lowfloor", "E_avg_A_B", "F_avg_A_D"]:
    for i in iso:
        p = OOF[name][i]; yi, fci = y[i], fc[i]
        labd = ~np.isnan(yi) & ~np.isnan(lo_col[i]) & ~np.isnan(hi_col[i])
        t, l, h = pseudo[i]; mm = ~np.isnan(t)
        lab = ~np.isnan(yi); inact = np.isnan(yi) & (fci > -0.3)
        rows.append(dict(model=name, isoform=i,
                         ST_RAE_labeled=st_rae(yi[labd], p[labd], lo_col[i][labd], hi_col[i][labd]),
                         ST_RAE_pseudo=st_rae(t[mm], p[mm], l[mm], h[mm]),
                         frac_inact_above_3p5=(p[inact] > 3.5).mean(),
                         labeled_bias=(p[lab] - yi[lab]).mean()))
R4 = pd.DataFrame(rows).round(3)
print("\n", R4.to_string(index=False))
print("\nmacro mean (lower ST-RAE is better):")
print(R4.groupby("model")[["ST_RAE_labeled", "ST_RAE_pseudo", "frac_inact_above_3p5", "labeled_bias"]].mean().round(3))

CYP1A2 done | mean floor = 2.24
CYP2C9 done | mean floor = 2.55
CYP2D6 done | mean floor = 2.14
CYP3A4 done | mean floor = 2.26

          model isoform  ST_RAE_labeled  ST_RAE_pseudo  frac_inact_above_3p5  labeled_bias
A_labeled_only  CYP1A2           0.848          0.874                 0.918        -0.013
A_labeled_only  CYP2C9           0.695          0.573                 0.869        -0.000
A_labeled_only  CYP2D6           0.985          0.638                 0.994        -0.001
A_labeled_only  CYP3A4           0.511          0.581                 0.454         0.005
   B_soft_w0.5  CYP1A2           1.281          0.606                 0.361        -0.572
   B_soft_w0.5  CYP2C9           0.782          0.493                 0.546        -0.203
   B_soft_w0.5  CYP2D6           1.235          0.781                 0.922        -0.382
   B_soft_w0.5  CYP3A4           0.479          0.518                 0.454         0.008
D_exp_lowfloor  CYP1A2           1.318          0.693       

In [10]:
mols_te = [Chem.MolFromSmiles(s) for s in te.SMILES]
assert all(m is not None for m in mols_te), "unparsable test SMILES"
fp_te = np.array([gen.GetCountFingerprintAsNumPy(m) for m in mols_te], dtype=np.float32)
desc_te = np.array([[f(m) for _, f in Descriptors.descList] for m in mols_te], dtype=np.float64)
desc_te = np.clip(np.nan_to_num(desc_te, nan=0, posinf=1e6, neginf=-1e6), -1e6, 1e6).astype(np.float32)
X_te = np.hstack([fp_te, desc_te])

sub = te[["SMILES", "Molecule_Name"]].copy()
for i in iso:
    yi, fci = y[i], fc[i]
    lab = np.where(~np.isnan(yi))[0]
    predA = lgb.LGBMRegressor(**params).fit(X[lab], yi[lab]).predict(X_te)
    okf = lab[~np.isnan(fci[lab])]
    ir = IsotonicRegression(increasing=False, out_of_bounds="clip").fit(fci[okf], yi[okf])
    unl = np.where(np.isnan(yi) & ~np.isnan(fci))[0]
    Xs = np.vstack([X[lab], X[unl]])
    ys = np.concatenate([yi[lab], ir.predict(fci[unl])])
    w = np.concatenate([np.ones(len(lab)), np.full(len(unl), 0.5)])
    predB = lgb.LGBMRegressor(**params).fit(Xs, ys, sample_weight=w).predict(X_te)
    sub[pc[i]] = 0.5 * (predA + predB)
    print(i, "done")

# format checks
cols = ["SMILES", "Molecule_Name"] + [pc[i] for i in iso]
assert list(sub.columns) == cols
assert sub.shape == (750, 6)
assert sub.Molecule_Name.is_unique
assert np.isfinite(sub[[pc[i] for i in iso]].values).all()
print("format checks passed:", sub.shape)
print(sub[[pc[i] for i in iso]].describe().round(2).loc[["mean", "min", "max"]])

sub.to_csv("cyp_submission_E.csv", index=False)
from google.colab import files
files.download("cyp_submission_E.csv")

CYP1A2 done
CYP2C9 done
CYP2D6 done
CYP3A4 done
format checks passed: (750, 6)
      CYP1A2_pIC50_direct_inhibition  CYP2C9_pIC50_direct_inhibition  \
mean                            4.63                            4.66   
min                             2.53                            2.66   
max                             6.48                            6.20   

      CYP2D6_pIC50_direct_inhibition  CYP3A4_pIC50_direct_inhibition  
mean                            4.46                            4.53  
min                             3.28                            1.88  
max                             6.16                            6.27  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
import warnings; warnings.filterwarnings("ignore")

def make_interval_obj(lo, hi, delta=0.5):
    def obj(y_true, y_pred):
        over = np.clip(y_pred - hi, 0, None)
        under = np.clip(lo - y_pred, 0, None)
        d = over + under
        s = np.sqrt(1 + (d / delta) ** 2)
        g = np.where(over > 0, d / s, np.where(under > 0, -d / s, 0.0))
        h = np.where(d > 0, s ** -3, 0.2)
        return g, h
    return obj

OOF["G_interval"] = {i: np.full(len(U), np.nan) for i in iso}
for i in iso:
    yi = y[i]; lo, hi = lo_col[i], hi_col[i]
    ok_all = ~np.isnan(yi) & ~np.isnan(lo) & ~np.isnan(hi)
    for tr_idx, te_idx in kf.split(U):
        idx = tr_idx[ok_all[tr_idx]]
        init = np.median(yi[idx])
        mid = (lo[idx] + hi[idx]) / 2
        m_ = lgb.LGBMRegressor(objective=make_interval_obj(lo[idx], hi[idx]), **params)
        m_.fit(X[idx], mid, init_score=np.full(len(idx), init))
        OOF["G_interval"][i][te_idx] = m_.predict(X[te_idx]) + init
    print(i, "done")

OOF["H_avg_G_B"] = {i: 0.5 * (OOF["G_interval"][i] + OOF["B_soft_w0.5"][i]) for i in iso}

rows = []
for name in ["A_labeled_only", "B_soft_w0.5", "E_avg_A_B", "G_interval", "H_avg_G_B"]:
    for i in iso:
        p = OOF[name][i]; yi, fci = y[i], fc[i]
        labd = ~np.isnan(yi) & ~np.isnan(lo_col[i]) & ~np.isnan(hi_col[i])
        t, l, h = pseudo[i]; mm = ~np.isnan(t)
        lab = ~np.isnan(yi); inact = np.isnan(yi) & (fci > -0.3)
        rows.append(dict(model=name, isoform=i,
                         ST_RAE_labeled=st_rae(yi[labd], p[labd], lo_col[i][labd], hi_col[i][labd]),
                         ST_RAE_pseudo=st_rae(t[mm], p[mm], l[mm], h[mm]),
                         frac_inact_above_3p5=(p[inact] > 3.5).mean(),
                         labeled_bias=(p[lab] - yi[lab]).mean()))
R5 = pd.DataFrame(rows).round(3)
print("\n", R5.to_string(index=False))
print("\nmacro mean (lower ST-RAE is better):")
print(R5.groupby("model")[["ST_RAE_labeled", "ST_RAE_pseudo", "frac_inact_above_3p5", "labeled_bias"]].mean().round(3))

CYP1A2 done
CYP2C9 done
CYP2D6 done
CYP3A4 done

          model isoform  ST_RAE_labeled  ST_RAE_pseudo  frac_inact_above_3p5  labeled_bias
A_labeled_only  CYP1A2           0.848          0.874                 0.918        -0.013
A_labeled_only  CYP2C9           0.695          0.573                 0.869        -0.000
A_labeled_only  CYP2D6           0.985          0.638                 0.994        -0.001
A_labeled_only  CYP3A4           0.511          0.581                 0.454         0.005
   B_soft_w0.5  CYP1A2           1.281          0.606                 0.361        -0.572
   B_soft_w0.5  CYP2C9           0.782          0.493                 0.546        -0.203
   B_soft_w0.5  CYP2D6           1.235          0.781                 0.922        -0.382
   B_soft_w0.5  CYP3A4           0.479          0.518                 0.454         0.008
     E_avg_A_B  CYP1A2           0.973          0.605                 0.708        -0.292
     E_avg_A_B  CYP2C9           0.692          0.

In [12]:
import warnings; warnings.filterwarnings("ignore")
TEST = {k: {} for k in ["A", "B", "G"]}
for i in iso:
    yi, fci = y[i], fc[i]; lo, hi = lo_col[i], hi_col[i]
    lab = np.where(~np.isnan(yi))[0]
    TEST["A"][i] = lgb.LGBMRegressor(**params).fit(X[lab], yi[lab]).predict(X_te)
    okf = lab[~np.isnan(fci[lab])]
    ir = IsotonicRegression(increasing=False, out_of_bounds="clip").fit(fci[okf], yi[okf])
    unl = np.where(np.isnan(yi) & ~np.isnan(fci))[0]
    Xs = np.vstack([X[lab], X[unl]])
    ys = np.concatenate([yi[lab], ir.predict(fci[unl])])
    w = np.concatenate([np.ones(len(lab)), np.full(len(unl), 0.5)])
    TEST["B"][i] = lgb.LGBMRegressor(**params).fit(Xs, ys, sample_weight=w).predict(X_te)
    idx = np.where(~np.isnan(yi) & ~np.isnan(lo) & ~np.isnan(hi))[0]
    init = np.median(yi[idx]); mid = (lo[idx] + hi[idx]) / 2
    g = lgb.LGBMRegressor(objective=make_interval_obj(lo[idx], hi[idx]), **params)
    g.fit(X[idx], mid, init_score=np.full(len(idx), init))
    TEST["G"][i] = g.predict(X_te) + init
    print(i, "done")

comp = te[["SMILES", "Molecule_Name"]].copy()
for k in TEST:
    for i in iso:
        comp[f"{k}_{i}"] = TEST[k][i]
comp.to_csv("cyp_test_components.csv", index=False)

def build(name, fn):
    s = te[["SMILES", "Molecule_Name"]].copy()
    for i in iso:
        s[pc[i]] = fn(i)
    assert list(s.columns) == ["SMILES", "Molecule_Name"] + [pc[i] for i in iso]
    assert s.shape == (750, 6) and s.Molecule_Name.is_unique
    assert np.isfinite(s[[pc[i] for i in iso]].values).all()
    s.to_csv(name, index=False)
    return s

sE = build("cyp_submission_E.csv", lambda i: 0.5 * (TEST["A"][i] + TEST["B"][i]))
sH = build("cyp_submission_H.csv", lambda i: 0.5 * (TEST["G"][i] + TEST["B"][i]))
print("format checks passed")
for i in iso:
    print(i, "corr(E,H) =", round(np.corrcoef(sE[pc[i]], sH[pc[i]])[0, 1], 3),
          "| mean E/H =", round(sE[pc[i]].mean(), 2), round(sH[pc[i]].mean(), 2))

from google.colab import files
for f in ["cyp_submission_H.csv", "cyp_test_components.csv"]:
    files.download(f)

CYP1A2 done
CYP2C9 done
CYP2D6 done
CYP3A4 done
format checks passed
CYP1A2 corr(E,H) = 0.974 | mean E/H = 4.63 4.72
CYP2C9 corr(E,H) = 0.973 | mean E/H = 4.66 4.76
CYP2D6 corr(E,H) = 0.935 | mean E/H = 4.46 4.53
CYP3A4 corr(E,H) = 0.99 | mean E/H = 4.53 4.63


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
sA = te[["SMILES", "Molecule_Name"]].copy()
for i in iso:
    sA[pc[i]] = TEST["A"][i]
assert list(sA.columns) == ["SMILES", "Molecule_Name"] + [pc[i] for i in iso]
assert sA.shape == (750, 6) and sA.Molecule_Name.is_unique
assert np.isfinite(sA[[pc[i] for i in iso]].values).all()
sA.to_csv("cyp_submission_A.csv", index=False)
print("A file ready; mean per isoform:", sA[[pc[i] for i in iso]].mean().round(2).to_dict())
from google.colab import files
files.download("cyp_submission_A.csv")

A file ready; mean per isoform: {'CYP1A2_pIC50_direct_inhibition': 4.96, 'CYP2C9_pIC50_direct_inhibition': 4.75, 'CYP2D6_pIC50_direct_inhibition': 4.65, 'CYP3A4_pIC50_direct_inhibition': 4.55}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
import warnings; warnings.filterwarnings("ignore")
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

rows = []
for i in iso:
    fci = fc[i]
    scr = np.where(~np.isnan(fci))[0]
    lab_inh = (fci[scr] <= -0.3).astype(int)
    p_te = LGBMClassifier(**params).fit(X[scr], lab_inh).predict_proba(X_te)[:, 1]
    p_tr = np.full(len(U), np.nan)
    for a, b in kf.split(U):
        s = a[~np.isnan(fci[a])]
        c = LGBMClassifier(**params).fit(X[s], (fci[s] <= -0.3).astype(int))
        p_tr[b] = c.predict_proba(X[b])[:, 1]
    m = ~np.isnan(fci)
    rows.append(dict(isoform=i,
                     train_actual_inhibitor_rate=lab_inh.mean(),
                     train_mean_P_inhibit_oof=np.nanmean(p_tr[m]),
                     test_mean_P_inhibit=p_te.mean(),
                     train_frac_P_below_0p3=np.mean(p_tr[m] < 0.3),
                     test_frac_P_below_0p3=np.mean(p_te < 0.3)))
print(pd.DataFrame(rows).round(3).to_string(index=False))

Xall = np.vstack([X, X_te]); dom = np.r_[np.zeros(len(X)), np.ones(len(X_te))]
pd_ = cross_val_predict(LGBMClassifier(**params), Xall, dom,
                        cv=StratifiedKFold(5, shuffle=True, random_state=0),
                        method="predict_proba")[:, 1]
print("\ntrain-vs-test domain classifier AUC:", round(roc_auc_score(dom, pd_), 3))

isoform  train_actual_inhibitor_rate  train_mean_P_inhibit_oof  test_mean_P_inhibit  train_frac_P_below_0p3  test_frac_P_below_0p3
 CYP1A2                        0.557                     0.570                0.673                   0.279                  0.189
 CYP2C9                        0.716                     0.750                0.795                   0.155                  0.124
 CYP2D6                        0.378                     0.351                0.406                   0.546                  0.487
 CYP3A4                        0.864                     0.912                0.909                   0.042                  0.049

train-vs-test domain classifier AUC: 0.949


In [15]:
tl = pd_[:len(X)]
print("training compounds with test-likeness > 0.5:", int((tl > 0.5).sum()),
      "| > 0.3:", int((tl > 0.3).sum()), "| of", len(tl))
subsets = {"all": np.ones(len(U), bool),
           "top25%": tl >= np.quantile(tl, 0.75),
           "top10%": tl >= np.quantile(tl, 0.90)}

rows = []
for name in ["A_labeled_only", "B_soft_w0.5", "E_avg_A_B", "G_interval", "H_avg_G_B"]:
    for sname, smask in subsets.items():
        st, mae, ns = [], [], []
        for i in iso:
            p, yi = OOF[name][i], y[i]
            m = smask & ~np.isnan(yi) & ~np.isnan(lo_col[i]) & ~np.isnan(hi_col[i])
            st.append(st_rae(yi[m], p[m], lo_col[i][m], hi_col[i][m]))
            mae.append(np.mean(np.abs(p[m] - yi[m]))); ns.append(int(m.sum()))
        rows.append(dict(model=name, subset=sname, macro_ST_RAE=np.mean(st),
                         macro_MAE=np.mean(mae), min_n=min(ns)))
print(pd.DataFrame(rows).round(3).to_string(index=False))

print("\nlabeled compounds per subset: mean pIC50 (n)")
for sname, smask in subsets.items():
    print(sname, {i: (round(float(np.nanmean(y[i][smask])), 2),
                      int((smask & ~np.isnan(y[i])).sum())) for i in iso})

training compounds with test-likeness > 0.5: 107 | > 0.3: 155 | of 4905
         model subset  macro_ST_RAE  macro_MAE  min_n
A_labeled_only    all         0.760      0.577   1285
A_labeled_only top25%         0.799      0.604    334
A_labeled_only top10%         0.841      0.652    128
   B_soft_w0.5    all         0.944      0.643   1285
   B_soft_w0.5 top25%         0.979      0.673    334
   B_soft_w0.5 top10%         0.987      0.711    128
     E_avg_A_B    all         0.791      0.582   1285
     E_avg_A_B top25%         0.834      0.611    334
     E_avg_A_B top10%         0.867      0.655    128
    G_interval    all         0.790      0.620   1285
    G_interval top25%         0.802      0.638    334
    G_interval top10%         0.818      0.662    128
     H_avg_G_B    all         0.751      0.581   1285
     H_avg_G_B top25%         0.783      0.606    334
     H_avg_G_B top10%         0.812      0.640    128

labeled compounds per subset: mean pIC50 (n)
all {'CYP1A2': (4.

In [16]:
import warnings; warnings.filterwarnings("ignore")
from scipy.stats import rankdata
r = rankdata(tl) / len(tl)

for g_ in [1.5, 3.0]:
    wts = np.exp(g_ * r)
    nA, nG = f"A_w{g_}", f"G_w{g_}"
    OOF[nA] = {i: np.full(len(U), np.nan) for i in iso}
    OOF[nG] = {i: np.full(len(U), np.nan) for i in iso}
    for i in iso:
        yi = y[i]; lo, hi = lo_col[i], hi_col[i]
        for tr_idx, te_idx in kf.split(U):
            lab = tr_idx[~np.isnan(yi[tr_idx])]
            OOF[nA][i][te_idx] = lgb.LGBMRegressor(**params).fit(
                X[lab], yi[lab], sample_weight=wts[lab]).predict(X[te_idx])
            idx = tr_idx[~np.isnan(yi[tr_idx]) & ~np.isnan(lo[tr_idx]) & ~np.isnan(hi[tr_idx])]
            init = np.median(yi[idx]); mid = (lo[idx] + hi[idx]) / 2
            gm = lgb.LGBMRegressor(objective=make_interval_obj(lo[idx], hi[idx]), **params)
            gm.fit(X[idx], mid, init_score=np.full(len(idx), init), sample_weight=wts[idx])
            OOF[nG][i][te_idx] = gm.predict(X[te_idx]) + init
    OOF[f"H_w{g_}"] = {i: 0.5 * (OOF[nG][i] + OOF["B_soft_w0.5"][i]) for i in iso}
    print("gamma", g_, "done")

rows = []
for name in ["A_labeled_only", "G_interval", "H_avg_G_B",
             "A_w1.5", "G_w1.5", "H_w1.5", "A_w3.0", "G_w3.0", "H_w3.0"]:
    for sname, smask in subsets.items():
        st, mae = [], []
        for i in iso:
            p, yi = OOF[name][i], y[i]
            m = smask & ~np.isnan(yi) & ~np.isnan(lo_col[i]) & ~np.isnan(hi_col[i])
            st.append(st_rae(yi[m], p[m], lo_col[i][m], hi_col[i][m]))
            mae.append(np.mean(np.abs(p[m] - yi[m])))
        rows.append(dict(model=name, subset=sname, macro_ST_RAE=np.mean(st), macro_MAE=np.mean(mae)))
T = pd.DataFrame(rows).round(3)
print(T.pivot(index="model", columns="subset", values="macro_ST_RAE")[["all", "top25%", "top10%"]])
print()
print(T.pivot(index="model", columns="subset", values="macro_MAE")[["all", "top25%", "top10%"]])

gamma 1.5 done
gamma 3.0 done
subset            all  top25%  top10%
model                                
A_labeled_only  0.760   0.799   0.841
A_w1.5          0.765   0.811   0.836
A_w3.0          0.761   0.804   0.844
G_interval      0.790   0.802   0.818
G_w1.5          0.790   0.802   0.818
G_w3.0          0.790   0.802   0.818
H_avg_G_B       0.751   0.783   0.812
H_w1.5          0.751   0.783   0.812
H_w3.0          0.751   0.783   0.812

subset            all  top25%  top10%
model                                
A_labeled_only  0.577   0.604   0.652
A_w1.5          0.578   0.609   0.652
A_w3.0          0.576   0.607   0.656
G_interval      0.620   0.638   0.662
G_w1.5          0.620   0.638   0.662
G_w3.0          0.620   0.638   0.662
H_avg_G_B       0.581   0.606   0.640
H_w1.5          0.581   0.606   0.640
H_w3.0          0.581   0.606   0.640


In [17]:
!git clone -q https://github.com/OpenADMET/CYP-Challenge-Tutorial.git
!ls -la CYP-Challenge-Tutorial/validation
!ls -la *.csv
import glob
for f in sorted(glob.glob("CYP-Challenge-Tutorial/validation/*.py")):
    print("=" * 20, f)
    print(open(f).read()[:7000])

total 36
drwxr-xr-x 2 root root  4096 Oct 12 07:02 .
drwxr-xr-x 9 root root  4096 Oct 12 07:02 ..
-rw-r--r-- 1 root root  3051 Oct 12 07:02 activity_validation.py
-rw-r--r-- 1 root root  3974 Oct 12 07:02 extract_lig_sdf.py
-rw-r--r-- 1 root root   196 Oct 12 07:02 __init__.py
-rw-r--r-- 1 root root 10700 Oct 12 07:02 structure_validation.py
-rw-r--r-- 1 root root  4046 Oct 12 07:02 tdi_validation.py
-rw-r--r-- 1 root root  99363 Oct 12 06:33 cyp_submission_A.csv
-rw-r--r-- 1 root root  99350 Oct 12 06:30 cyp_submission_E.csv
-rw-r--r-- 1 root root  99315 Oct 12 06:30 cyp_submission_H.csv
-rw-r--r-- 1 root root 207937 Oct 12 06:30 cyp_test_components.csv
==================== CYP-Challenge-Tutorial/validation/__init__.py
from .activity_validation import validate_activity_submission
from .tdi_validation import validate_tdi_submission

__all__ = [
    "validate_activity_submission",
    "validate_tdi_submission",
]

==================== CYP-Challenge-Tutorial/validation/activity_validatio

In [18]:
import sys
from pathlib import Path
sys.path.insert(0, "CYP-Challenge-Tutorial")
from validation import validate_activity_submission

test_ids = set(te.Molecule_Name)
for f in ["cyp_submission_H.csv", "cyp_submission_E.csv"]:
    ok, errs = validate_activity_submission(Path(f), expected_ids=test_ids)
    s = pd.read_csv(f)
    same_order = bool((s.Molecule_Name.values == te.Molecule_Name.values).all())
    same_smiles = bool((s.SMILES.values == te.SMILES.values).all())
    print(f, "| valid:", ok, "| errors:", errs,
          "| same ID order as test:", same_order, "| SMILES match test:", same_smiles)

cyp_submission_H.csv | valid: True | errors: [] | same ID order as test: True | SMILES match test: True
cyp_submission_E.csv | valid: True | errors: [] | same ID order as test: True | SMILES match test: True


In [19]:
import pickle, zipfile, os, glob
from google.colab import files

# 1) Save in-memory results, which vanish if the session resets
names = ["OOF", "TEST", "PINH", "FLOOR", "pseudo", "pd_", "params"]
state = {n: globals()[n] for n in names if n in globals()}
if "U" in globals():
    state["compound_names"] = U.Molecule_Name.values
with open("cyp_session_state.pkl", "wb") as f:
    pickle.dump(state, f)
print("saved in-memory objects:", list(state.keys()))

# 2) Zip all files in the session folder
bundle = "cyp_challenge_files.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for pat in ["*.csv", "*.pkl", "*.py", "*.txt", "*.md", "*.json"]:
        for f in glob.glob(pat):
            z.write(f)
            print("added", f, os.path.getsize(f) // 1024, "KB")

# 3) Download
files.download(bundle)

saved in-memory objects: ['OOF', 'TEST', 'PINH', 'FLOOR', 'pseudo', 'pd_', 'params', 'compound_names']
added cyp_test_components.csv 203 KB
added cyp_submission_E.csv 97 KB
added cyp_submission_A.csv 97 KB
added cyp_submission_H.csv 96 KB
added cyp_session_state.pkl 3409 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
from google.colab import drive, files
drive.mount('/content/drive')
import glob
nbs = glob.glob('/content/drive/MyDrive/**/*.ipynb', recursive=True)
for k, p in enumerate(nbs): print(k, p)

Mounted at /content/drive
0 /content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
1 /content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
2 /content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
3 /content/drive/MyDrive/Colab Notebooks/Untitled3.ipynb
4 /content/drive/MyDrive/Colab Notebooks/01_CADD_Chemistry_Engine.ipynb
5 /content/drive/MyDrive/Colab Notebooks/02_ADMET_Solubility_Model.ipynb
6 /content/drive/MyDrive/Colab Notebooks/03_ADMET_BBB_Model.ipynb
7 /content/drive/MyDrive/Colab Notebooks/01_multidisease_platform_setup.ipynb
8 /content/drive/MyDrive/Colab Notebooks/Untitled4.ipynb
9 /content/drive/MyDrive/Colab Notebooks/Untitled5.ipynb
10 /content/drive/MyDrive/Colab Notebooks/Untitled6.ipynb
11 /content/drive/MyDrive/Colab Notebooks/Copy of AD_pTau217_MIP_InSilico_Study.ipynb
12 /content/drive/MyDrive/Colab Notebooks/AD_pTau217_MIP_InSilico_Study.ipynb
13 /content/drive/MyDrive/Colab Notebooks/Untitled7.ipynb
14 /content/drive/MyDrive/Colab Notebooks/cyp_challenge.ipynb
1

In [ ]:
files.download(nbs[0])